# Drop-out experiments
This notebook focuses on the analyses of the C6 models.

# Imports and Load Individual Models

In [ ]:
# basic
import os
import pandas as pd
import re
import numpy as np
import copy

# see functions.py for a collection of self-written functions for analysing and visualising my models
from functions import *

# cobra & micom
from cobra.io import read_sbml_model, write_sbml_model
from cobra.flux_analysis import pfba, flux_variability_analysis
from cobra.medium import minimal_medium
from cobra.exceptions import Infeasible, OptimizationError
from micom import Community
from micom.problems import cooperative_tradeoff
from micom import load_pickle

# plots
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, LogNorm
import seaborn as sns
import matplotlib.colors as mcolors

# PCA
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Load SBML Models - individuals

In [ ]:
models_path = "../Models/02_afterCuration/"
models = {}
for model_name in (f for f in os.listdir(models_path) if f.endswith(".xml")):
    model = read_sbml_model(f"{models_path}/{model_name}")
    model.solver = "cplex"
    name = str(model_name[:3])
    models[name] = model

models = {key: models[key] for key in sorted(models.keys())}  # sorts the dictionary alphabetically
AA1, AA2, AA3, AA4, AA5, AA6, AA7 = [models[f"AA{i}"] for i in range(1, 8)]

# Functions

In [ ]:
def e_to_m(medium):
    if isinstance(medium, dict):
        # Convert reaction IDs ending in '_e' to '_m'
        medium_dict = {k.removesuffix('_e') + '_m' if k.endswith('_e') else k: v for k, v in medium.items()}
        return medium_dict

    elif isinstance(medium, pd.DataFrame):
        # Convert DataFrame to a dict with converted reaction IDs
        medium_dict = {rxn.removesuffix('_e') + '_m' if rxn.endswith('_e') else rxn: bound for rxn, bound in
                       zip(medium["reaction"], medium["bound"])}
        return medium_dict

    else:
        return "Medium is wrong data type"

In [ ]:
def heatmap_growth_dropouts(models, medium, medium_name, save_path=None):
    growth_data = {}

    for name, syncom in models.items():
        _, growth = community_pfba(syncom, medium)
        if growth is not None:
            growth_data[name] = growth

    growth_df = pd.DataFrame(growth_data).transpose()

    epsilon = 1e-6
    growth_df = growth_df.applymap(lambda x: 0 if abs(x) < epsilon else x)

    # --- Rename strains (columns) ---
    strain_map = {
        "AA1": "Sma",
        "AA2": "Bpi",
        "AA3": "Cpu",
        "AA4": "Elu",
        "AA5": "Cin",
        "AA6": "Hro",
        "AA7": "Ppu"
    }
    growth_df = growth_df.rename(columns=strain_map)

    # --- Rename communities (rows) ---
    community_map = {
        "C7_Community": "C7",
        "syncom_drop1_2": "C6-Sma",
        "syncom_drop2_2": "C6-Bpi",
        "syncom_drop3_2": "C6-Cpu",
        "syncom_drop4_2": "C6-Elu",
        "syncom_drop5_2": "C6-Cin",
        "syncom_drop6_2": "C6-Hro",
        "syncom_drop7_2": "C6-Ppu"
    }
    growth_df = growth_df.rename(index=community_map)
    #return growth_df

    # Desired plotting order
    strain_order = ["Sma", "Bpi", "Cpu", "Elu", "Cin", "Hro", "Ppu"]
    community_order = ["C7", "C6-Sma", "C6-Bpi", "C6-Cpu", "C6-Elu", "C6-Cin", "C6-Hro", "C6-Ppu"]

    plt.figure(figsize=(10, 6))
    ax = sns.heatmap(
        growth_df.loc[community_order, strain_order].T,  # transpose here!
        fmt=".3f",
        annot=True,
        cmap="Reds",
        cbar_kws={'label': 'Flux through Biomass\n[mmol/(gDW h)]'}
    )

    ax.set_xticklabels(community_order, rotation=45, ha="right")
    ax.set_yticklabels(strain_order, rotation=0)

    plt.title(f"Growth per strain across community model variants (including full community)\non {medium_name} medium")
    plt.xlabel("Community variant")
    plt.ylabel("Strains")
    plt.tight_layout()

    if save_path is not None:
        file_name = f"drop_outs_growth_{medium_name}.svg"
        plt.savefig(f"../Figures/{file_name}", format="svg", bbox_inches="tight")
    plt.show()

    return growth_df

# Medium

In [ ]:
m9 = pd.read_csv("../Datasets/Medium/medium_M9.csv", sep=";")
m9_rxn = list(m9["reaction"])
maize_root_medium = pd.read_csv("../Datasets/Medium/medium_combined_Krumbach_M9.csv", sep=";", quotechar='"')
ecoplate = pd.read_csv("../Datasets/Medium/medium_ecoplate_carbon.csv", sep=";", quotechar='"')
ecoplate_rxns = [safe_parse(x) for x in ecoplate["reaction"]]
ecoplate_short = pd.read_csv("../Datasets/Medium/medium_ecoplate_carbon_shortened.csv", sep=";", quotechar='"')
ecoplate_rxns_short = [safe_parse(x) for x in ecoplate_short["reaction"]]

# Create C7 Community Model with MICOM

In [ ]:
taxonomy = pd.read_csv("../Datasets/community_model_gap_filled.csv", sep=";")
syncom = Community(taxonomy.iloc[:, :-1], name="synthetic seven member bacterial community", id="C7", solver="cplex")

# change lower bound of Exchange reactions (EX between bacteria and medium)
for rxn in syncom.reactions.query("^EX_"):
    rxn.lower_bound = -1000
taxonomy_noS = pd.read_csv("../Datasets/community_model_noSucrose.csv", sep=";")
syncom_half = Community(taxonomy_noS.iloc[:, :-1], name="synthetic seven member bacterial community", id="C7_half",
                        solver="cplex")

for rxn in syncom_half.reactions.query("^EX_"):
    rxn.lower_bound = -1000

# Create Drop-out Communities
From our C7 community we want to exclude one bacterium at a time making it a C6 community

In [ ]:
# Load Models - Gap Filled but NO Sucrose Fixes - drop outs
models_path = "../Models/12_1_noSucroseFix/Drop_outs_pickle"
drops_noS = {}
for model_name in (f for f in os.listdir(models_path) if f.endswith(".pickle")):
    model = load_pickle(f"{models_path}/{model_name}")
    model.solver = "cplex"
    name = str(model_name[:2] + "_noS")
    drops_noS[name] = model

drops_noS = {key: drops_noS[key] for key in sorted(drops_noS.keys())}  # sorts the dictionary alphabetically
S1_noS, S2_noS, S3_noS, S4_noS, S5_noS, S6_noS, S7_noS = [drops_noS[f"S{i}_noS"] for i in range(1, 8)]
syncoms_half = {}

for i in range(len(taxonomy_noS)):
    cut_tax = taxonomy_noS.drop(index=taxonomy_noS.index[i])
    name = f"syncom_drop{i + 1}_2"

    com = Community(
        cut_tax.iloc[:, :-1],
        name="synthetic six member bacterial community",
        id=name,
        solver="cplex")

    # Set EX reaction bounds for this syncom
    for rxn in com.reactions.query("^EX_"):
        rxn.lower_bound = -1000  # make exchange freely available

    syncoms_half[name] = com

In [ ]:
S1_2, S2_2, S3_2, S4_2, S5_2, S6_2, S7_2 = [syncoms_half[f"syncom_drop{i + 1}_2"] for i in range(7)]
# Load Models - Gap Filled but NO Sucrose Fixes - drop outs
models_path = "../Models/12_gap_filled/Drop_outs_pickle"
drops = {}
for model_name in (f for f in os.listdir(models_path) if f.endswith(".pickle")):
    model = load_pickle(f"{models_path}/{model_name}")
    model.solver = "cplex"
    name = str(model_name[:2])
    drops[name] = model

drops = {key: drops[key] for key in sorted(drops.keys())}  # sorts the dictionary alphabetically
S1, S2, S3, S4, S5, S6, S7 = [drops[f"S{i}"] for i in range(1, 8)]
syncoms = {}

for i in range(len(taxonomy)):
    cut_tax = taxonomy.drop(index=taxonomy.index[i])
    name = f"syncom_drop{i + 1}"

    com = Community(
        cut_tax.iloc[:, :-1],
        name="synthetic six member bacterial community",
        id=name,
        solver="cplex")

    # Set EX reaction bounds for this syncom
    for rxn in com.reactions.query("^EX_"):
        rxn.lower_bound = -1000  # make exchange freely available

    syncoms[name] = com

In [ ]:
S1, S2, S3, S4, S5, S6, S7 = [syncoms[f"syncom_drop{i + 1}"] for i in range(7)]
all_models = {**syncoms, 'C7_Community': syncom}
all_models_half = {**drops_noS, 'C7_Community': syncom_half}

ATTENTION: "normal" saving with write_sbml_model doesnt work for MICOM Community model \
        I need to save them as pickles \
        see: https: // resendislab.github.io / micom / community.html

# Saving-and-loading-communities

In [ ]:
# save all drop-out models as file
#for i, model in enumerate(syncoms.values(), start=1):
#   path = f"/home/lisa/Dokumente/Programmierung/Models/12_gap_filled/Drop_outs/S{i}.xml"
#  write_sbml_model(model, path)
# save all drop-out models as pickles
for i, model in enumerate(syncoms_half.values(), start=1):
    path = f"../Models/12_1_noSucroseFix/Drop_outs_pickle/S{i}_noS.pickle"
    model.to_pickle(path)
# save all drop-out models as pickles
for i, model in enumerate(syncoms.values(), start=1):
    path = f"../Models/12_gap_filled/Drop_outs_pickle/S{i}.pickle"
    model.to_pickle(path)


In [ ]:
# Growth for all drop-out syncoms
test_medium(syncom, maize_root_medium)
for model in all_models.values():
    print(test_medium(model, maize_root_medium))

In [ ]:
del_suc = ["EX_sucr_e__AA1", "EX_sucr_e__AA3", "EX_sucr_e__AA5", "EX_sucr_e__AA6"]
for rxn_id in del_suc:
    if rxn_id in S7_2.reactions:
        S7_2.reactions.get_by_id(rxn_id).bounds = (0, 1000)
growth_krumbach_df = heatmap_growth_dropouts(all_models_half, maize_root_medium, "Krumbach", save_path=True)
growth_krumbach_df["row_sum"] = growth_krumbach_df.sum(axis=1)
s7_fluxes = convert_cooptradeoff_into_fluxes(S7, maize_root_medium)

In [ ]:
# pFBA in comparison to coop. tradeoff
for model in drops_noS.values():
    with model:
        change_medium(model, maize_root_medium)
        fluxes = pfba(model).fluxes
        print(model.id)
        print(fluxes[fluxes.index.str.contains("Growth")])
with S7:
    change_medium(S7, maize_root_medium)
    fluxes = pfba(S7).fluxes
fluxes[fluxes.index.str.contains("Growth")].sum()
with S7:
    change_medium(S7, maize_root_medium)
    fluxes2 = S7.cooperative_tradeoff(pfba=True, fluxes=True).fluxes.T
fluxes2[fluxes2.index.str.contains("Growth")].T
fluxes2[fluxes2.index.str.contains("Growth")].sum(axis=1)
with S7_2:
    change_medium(S7_2, maize_root_medium)
    s7_pfba_withS = pfba(S7_2).fluxes

In [ ]:
s7_fluxes_withS = convert_cooptradeoff_into_fluxes(S7_2, maize_root_medium)
s3_fluxes_withS = convert_cooptradeoff_into_fluxes(S3_2, maize_root_medium)
growth_krumbach_df = heatmap_growth_dropouts(all_models_half, maize_root_medium, "Krumbach")

growth_krumbach_df = heatmap_growth_dropouts(all_models, maize_root_medium, "Krumbach")

pca(growth_krumbach_df)

# Growth on Single Carbon Sources

In [ ]:
carbon_sources = ["EX_glc__D_e", "EX_sucr_e", "EX_xyl__D_e", ["EX_mal__L_e", "EX_mal__L_e"], "EX_pyr_e", "EX_succ_e",
                  ["EX_ala__L_e", "EX_ala__L_e"], "EX_glu__L_e", "EX_gly_e"]
carbon_df = create_medium(carbon_sources, m9_rxn, all_models, 10, carbon_only=True)

In [ ]:
metab_axis = ["Phenylethylamine", "Putrescine", "Glycyl-glutamic acid", "L-arginine", "Asparagine", "Phenylalanine",
                  "Serine", "Threonine", "Glycerol-phosphate", "Cellobiose", "Mannitol", "Xylose",
                  "Glucose-1-phosphate", "Acetyl-glucosamine", "4-Hydroxy benzoic acid", "Galacturonic acid",
                  "Malic acid", "2-Hydroxy-benzoic acid", "Lactose", "Butyric acid", "Water (Control)"]

In [ ]:
# Substrate Utilisation (BIOLOG Ecoplate)
#def create_medium(carbon_list, minimal_list, model_dict, medium_uptake_bound, carbon_only=False):
#TODO: war missing, maybe ist die FUnktion im functions.py

In [ ]:
ecoplate_df = create_medium(ecoplate_rxns, m9_rxn, all_models, 10, carbon_only=True, average="yes")
# takes around 9min

In [ ]:
# for nicer visualisation: better labels
ecoplate_df.index = metab_axis
ecoplate_df.columns = ["C6-Sma", "C6-Bpi", "C6-Cpu", "C6-Elu", "C6-Cin", "C6-Hro", "C6-Ppu", "C7"]
ecoplate_df = ecoplate_df.fillna(0)
# Create a heatmap
plt.figure(figsize=(10, 6))
sns.heatmap(ecoplate_df, annot=True, cmap="YlOrRd", cbar_kws={'label': 'Flux through Biomass [mmol/(gDW·h)]'})
# plt.title("")
plt.xlabel("Strain")
#plt.yticks(ticks=np.arange(len(metab_axis)), labels=metab_axis)
plt.ylabel("Additional Carbon Source to M9 Minimal Media")
#plt.savefig("../Figures/ecoplate.svg", format="svg", bbox_inches="tight")
plt.show()

In [ ]:
pca(ecoplate_df.T)

In [ ]:
ecoplate_df_highcarbon = create_medium(ecoplate_rxns, m9_rxn, all_models, 1000, carbon_only=True, average="yes")

In [ ]:
# for nicer visualisation: better labels
ecoplate_df_highcarbon.index = metab_axis
ecoplate_df_highcarbon.columns = ["C6-Sma", "C6-Bpi", "C6-Cpu", "C6-Elu", "C6-Cin", "C6-Hro", "C6-Ppu", "C7"]
ecoplate_df_highcarbon = ecoplate_df_highcarbon.fillna(0)
# Create a heatmap
plt.figure(figsize=(10, 6))
sns.heatmap(ecoplate_df_highcarbon, annot=True, cmap="YlOrRd",
            cbar_kws={'label': 'Flux through Biomass [mmol/(gDW·h)]'})
# plt.title("")
plt.xlabel("Strain")
#plt.yticks(ticks=np.arange(len(metab_axis)), labels=metab_axis)
plt.ylabel("Additional Carbon Source to M9 Minimal Media")
#plt.savefig("../Figures/ecoplate.svg", format="svg", bbox_inches="tight")
plt.show()

In [ ]:
pca(ecoplate_df_highcarbon.T)

In [ ]:
# Combine Heatmaps
growth_with_total = growth_krumbach_df.copy()

# First calculate the raw sum
growth_with_total["Total"] = growth_with_total.sum(axis=1)

In [ ]:
# Adjust Total based on the index
def adjust_total(row):
    if "drop" in row.name:
        return row["Total"] / 6
    elif row.name == "C7_Community":
        return row["Total"] / 7
    else:
        return row["Total"]  # Leave unchanged if none of the above

In [ ]:
# Apply the adjustment
growth_with_total["Total"] = growth_with_total.apply(adjust_total, axis=1)

In [ ]:
# Create a mask for the last row ("Total")
mask_top = np.zeros_like(growth_with_total.T, dtype=bool)
mask_top[-1, :] = True  # mask the last row to avoid color distortion
# Custom axis labels
custom_labels_y = ["C7", "C6-Sma", "C6-Bpi", "C6-Cpu", "C6-Elu", "C6-Cin", "C6-Hro", "C6-Ppu"]
custom_labels_x = ["Sma", "Bpi", "Cpu", "Elu", "Cin", "Hro", "Ppu", "Community mean"]
#metab_axis = ["Phenylethylamine", "Putrescine", "Glycyl-glutamic acid", "L-arginine", "Asparagine", "Phenylalanine", "Serine", "Threonine", "Glycerol-phosphate", "Cellobiose", "mannitol", "Xylose", "Glucose-1-phosphate", "Acetyl-glucosamine", "Galacturonic acid", "Malic acid", "2-Hydroxy-benzoic acid", "Lactose", "Butyric acid", "Water (Control)"]
metab_axis = ["Phenylethylamine", "Putrescine", "L-arginine", "Asparagine", "Phenylalanine", "Glycerol-phosphate",
              "Cellobiose", "Mannitol", "Xylose", "Glucose-1-phosphate", "Acetyl-glucosamine", "Galacturonic acid",
              "2-Hydroxy-benzoic acid", "Lactose", "Water (Control)"]

In [ ]:
fig, (ax1, ax2) = plt.subplots(
    1, 2,
    figsize=(22, 7),
    sharey=True,
    gridspec_kw={'width_ratios': [1, 1.7]}
)

vmin = 0
vmax = 45
color_map1 = "Reds"
color_map2 = "Blues"

# --- First heatmap: Growth ---
sns.heatmap(
    growth_with_total,
    ax=ax1,
    cmap=color_map1,
    annot=True,
    fmt=".1f",
    vmin=0,
    vmax=25,
    cbar=True,
    cbar_kws={'label': 'Flux through Biomass [mmol/(gDW·h)]'}
)
ax1.set_ylabel("Community Variant")
ax1.set_xlabel("Strains")
ax1.set_title("Growth across Community Variants dependent on medium and carbon sources", fontsize=14)
ax1.set_xticklabels(custom_labels_x, rotation=45, ha="right")
ax1.set_yticklabels(custom_labels_y, ha="right")

In [ ]:
# --- Second heatmap: Metabolite Uptake ---
res_df_clean = res_df_avg[[res_df_avg.columns[-1]] + list(res_df_avg.columns[:-1])]
res_df_clean = res_df_clean.drop(index="4hbz", errors="ignore").fillna(0)
res_df_clean_T = res_df_clean.T

heatmap = sns.heatmap(
    res_df_clean_T,
    ax=ax2,
    cmap=color_map2,
    vmin=0,
    vmax=30,
    annot=True,
    fmt=".1f",
    cbar=True,
    cbar_kws={'label': 'Flux through Biomass [mmol/(gDW·h)]'}
)
ax2.set_xlabel("Additional Carbon Source")
#ax2.set_title("Metabolite Uptake across Community Variants", fontsize=14)
ax2.set_xticklabels(metab_axis, rotation=45, ha="right")
ax2.set_yticklabels(custom_labels_y, ha="right")  # Y is shared with ax1

# --- Shared colorbar ---
#cbar_ax = fig.add_axes([0.93, 0.25, 0.02, 0.5])
#fig.colorbar(heatmap.get_children()[0], cax=cbar_ax, label="Flux through Biomass\n[mmol/(gDW·h)]")

# --- Global super title ---
#fig.suptitle("Growth across Community Variants on", fontsize=16, fontweight='bold', y=0.98, x=0.42)

# --- Individual titles for each subplot ---
ax1.set_title("(a) Growth across Community Variants on Maize Root Exudate", fontsize=13)
ax2.set_title("(b) Growth across Community Variants on M9 Minimal Medium", fontsize=13)

# --- Layout ---
#plt.tight_layout(rect=[0, 0, 0.92, 1])
fig.subplots_adjust(wspace=0.05)

# --- Save & Show ---
#plt.savefig("../Figures/combined_heatmap.svg", format="svg", bbox_inches="tight")
plt.show()

In [ ]:
# Standardise the data
scaler = StandardScaler()
growth_df_filled = res_df_clean.T.fillna(0)
scaled_growth = scaler.fit_transform(growth_df_filled)

# Perform PCA
pca = PCA(n_components=2)
pca_result = pca.fit_transform(scaled_growth)

# Create df for PCA result
pca_df = pd.DataFrame(pca_result, columns=["PC1", "PC2"], index=res_df_clean.T.index)

# Plot
plt.figure(figsize=(8, 6))
sns.scatterplot(x="PC1", y="PC2", data=pca_df, s=100)

# Annotate each point with the community name
for name, row in pca_df.iterrows():
    if name != "C7_Community":
        plt.text(row["PC1"] + 0.07, row["PC2"] - 0.04, name, fontsize=9)
    else:
        plt.text(row["PC1"] + -0.1, row["PC2"] + 0.1, name, fontsize=9)

# Add variance explained info
pc1_var = pca.explained_variance_ratio_[0] * 100
pc2_var = pca.explained_variance_ratio_[1] * 100
plt.xlabel(f"PC1 ({pc1_var:.1f}% variance)")
plt.ylabel(f"PC2 ({pc2_var:.1f}% variance)")

plt.title("PCA of Growth Profiles Across Community Variants")
#plt.xlim(-3, 3)
#plt.ylim(-3, 2)
plt.grid(True)
plt.tight_layout()
#plt.savefig("../Figures/drop_outs_ecoplate_pca.svg", format="svg", bbox_inches="tight")
plt.show()

# Uptakes and Secretions

## One community within

In [ ]:
heatmap_fluxes_withinCommunity(syncom, models, maize_root_medium, "uptake")
new_dict = {k: v for k, v in models.items() if k != "AA7"}
new_dict = {k: v for k, v in models.items() if k != "AA7"}
extra_all_up = heatmap_fluxes_withinCommunity(S7, new_dict, maize_root_medium_all, "uptake")
syncom.reactions.query("EX_thm")
get_rxn(AA4, "PFL")
get_rxn_unknown(models, "PFL")
get_met(AA4, "for_c")
heatmap_fluxes_withinCommunity(syncom, models, maize_root_medium, "secret")
extra_all = heatmap_fluxes_withinCommunity(S7, new_dict, maize_root_medium_all, "secret")

## across all drop out communities

In [ ]:
uptake_df_alldrops = uptake_secret_heatmap(all_models, "uptake", medium_combined_naveed, c7_all_ex="no")
secret_df_alldrops = uptake_secret_heatmap(all_models, "secret", medium_combined_naveed, c7_all_ex="no", vmin=0.01)

In [ ]:
# S7 Investigation
taxonomy = pd.read_csv("../Datasets/community_model_gap_filled.csv", sep=";")
cut_tax = taxonomy.drop(index=taxonomy.index[6])
name = f"syncom_drop7"

S7 = Community(
    cut_tax.iloc[:, :-1],
    name="synthetic six member bacterial community",
    id=name,
    solver="cplex")

# Set EX reaction bounds for this syncom
for rxn in S7.reactions.query("^EX_"):
    rxn.lower_bound = -1000
taxonomy = pd.read_csv("../Datasets/community_model_noSucrose.csv", sep=";")
cut_tax = taxonomy.drop(index=taxonomy.index[6])
name = f"syncom_drop7_noS"

S7_noS = Community(
    cut_tax.iloc[:, :-1],
    name="synthetic six member bacterial community",
    id=name,
    solver="cplex")

# Set EX reaction bounds for this syncom
for rxn in S7_noS.reactions.query("^EX_"):
    rxn.lower_bound = -1000
## Sucrose
with S7_2:
    S7_2.reactions.EX_sucr_e__AA6.knock_out()
    growth_val = test_medium(S7_2, maize_root_medium)
    print(growth_val)
growth_val = test_medium(S7, maize_root_medium)
print(growth_val)
growth_val_fixed = test_medium(S7_2, maize_root_medium)
print(growth_val_fixed)
sum(growth_val_fixed)
test_fractions_community(S7, maize_root_medium_all, drop_out="Ppu", medium_name="Krumbach + Allose")
test_fractions_community(S7, maize_root_medium, drop_out="Ppu", medium_name="Krumbach")
test_fractions_community(S7_noS, maize_root_medium, drop_out="Ppu", medium_name="Krumbach")
test_fractions_community(S7, medium_combined_naveed, drop_out="Ppu", medium_name="Naveed")
test_fractions_community(S7_noS, medium_combined_naveed, drop_out="Ppu", medium_name="Naveed")

### FVA
with S7:
    change_medium(S7, maize_root_medium)
    fva_s7_filled = flux_variability_analysis(S7)
#fva_s7_filled.to_csv("../FVA_DropOut7_GapFilled_Krumbach.csv", sep=";")
with S7_noS:
    change_medium(S7_noS, maize_root_medium)
    fva_s7_noSucrose = flux_variability_analysis(S7_noS)
#fva_s7_noSucrose.to_csv("../FVA_DropOut7_FixesButNotSucrose_Krumbach.csv", sep=";")
fva_s7_noSucrose = pd.read_csv("../FVA_DropOut7_FixesButNotSucrose_Krumbach.csv", sep=";", index_col=0)
fva_s7_filled = pd.read_csv("../FVA_DropOut7_GapFilled_Krumbach.csv", sep=";", index_col=0)
fva_s7_noSucrose = fva_s7_noSucrose.round(5)
fva_s7_filled = fva_s7_filled.round(5)

Check which reactions always have the same flux value(that is not zero), aka what are essential reactions

In [ ]:
# mask for rows where minimum == maximum and not 0
mask = (fva_s7_noSucrose["minimum"] == fva_s7_noSucrose["maximum"]) & (fva_s7_noSucrose["minimum"] != 0)

# filter rows
identical_rows = fva_s7_noSucrose[mask]

# print index and the identical value
for idx, row in identical_rows.iterrows():
    print(idx, row["minimum"])
# mask for rows where minimum == maximum and not 0
mask = (fva_s7_filled["minimum"] == fva_s7_filled["maximum"]) & (fva_s7_filled["minimum"] != 0)

# filter rows
identical_rows = fva_s7_filled[mask]

# print index and the identical value
for idx, row in identical_rows.iterrows():
    print(idx, row["minimum"])

get Reactions where min == max != 0 that are unique to that model /
--> what are differences between models \
--> are there other reactions active

In [ ]:
# mask for fixed nonzero fluxes
mask_filled = (fva_s7_filled["minimum"] == fva_s7_filled["maximum"]) & (fva_s7_filled["minimum"] != 0)
mask_noSucrose = (fva_s7_noSucrose["minimum"] == fva_s7_noSucrose["maximum"]) & (fva_s7_noSucrose["minimum"] != 0)

# extract indices and values
fixed_filled = fva_s7_filled[mask_filled]["minimum"]
fixed_noSucrose = fva_s7_noSucrose[mask_noSucrose]["minimum"]

# find unique reactions
unique_to_filled = fixed_filled.drop(index=fixed_noSucrose.index, errors="ignore").sort_index()
unique_to_noSucrose = fixed_noSucrose.drop(index=fixed_filled.index, errors="ignore").sort_index()

print("Unique to filled model:")
print(unique_to_filled)

print("\nUnique to noSucrose model:")
print(unique_to_noSucrose)

what reactions have zero flux in one model and non zero flux in the other

In [ ]:
# get common indices
common_idx = fva_s7_filled.index.intersection(fva_s7_noSucrose.index)

# restrict both dfs to common indices
fva_s7_filled_common = fva_s7_filled.loc[common_idx]
fva_s7_noSucrose_common = fva_s7_noSucrose.loc[common_idx]

# masks for 0/0 rows
mask_filled_zero = (fva_s7_filled_common["minimum"] == 0) & (fva_s7_filled_common["maximum"] == 0)
mask_noSucrose_zero = (fva_s7_noSucrose_common["minimum"] == 0) & (fva_s7_noSucrose_common["maximum"] == 0)

# Case 1: zero in filled but nonzero in noSucrose
case1 = fva_s7_noSucrose_common[mask_filled_zero & ~mask_noSucrose_zero][["minimum"]].copy()
#case1 = case1.rename(columns={"minimum": "nonzero_value"})
#case1 = case1.reset_index().rename(columns={"index": "reaction_id"})

# Case 2: zero in noSucrose but nonzero in filled
case2 = fva_s7_filled_common[mask_noSucrose_zero & ~mask_filled_zero][["minimum"]].copy()
#case2 = case2.rename(columns={"minimum": "nonzero_value"})
#case2 = case2.reset_index().rename(columns={"index": "reaction_id"})

print("Zero in filled but nonzero in noSucrose:")
print(case1)

print("\nZero in noSucrose but nonzero in filled:")
print(case2)
get unique reactions of each model and check if they have flux at all \
    --> are the sucrose reactions that i am deleting even used?? \
    --> apparently not ??
# get reactions unique to each df
unique_filled = fva_s7_filled.index.difference(fva_s7_noSucrose.index)
unique_noSucrose = fva_s7_noSucrose.index.difference(fva_s7_filled.index)

# filter df to only unique reactions
filled_unique_df = fva_s7_filled.loc[unique_filled]
noSucrose_unique_df = fva_s7_noSucrose.loc[unique_noSucrose]

# keep only rows where min/max are nonzero
filled_nonzero_unique = filled_unique_df[(filled_unique_df["minimum"] != 0) | (filled_unique_df["maximum"] != 0)].copy()
noSucrose_nonzero_unique = noSucrose_unique_df[
    (noSucrose_unique_df["minimum"] != 0) | (noSucrose_unique_df["maximum"] != 0)].copy()

# reset index to have reaction_id as a column
filled_nonzero_unique = filled_nonzero_unique.reset_index().rename(columns={"index": "reaction_id"})
noSucrose_nonzero_unique = noSucrose_nonzero_unique.reset_index().rename(columns={"index": "reaction_id"})

print("Unique reactions in filled with nonzero min/max:")
print(filled_nonzero_unique)

print("\nUnique reactions in noSucrose with nonzero min/max:")
print(noSucrose_nonzero_unique)

In [ ]:
with S7:
    change_medium(S7, maize_root_medium)
    s7_fluxies = pfba(S7).fluxes

    filtered = s7_fluxies[s7_fluxies.index.str.contains("Growth")]
    print(filtered)

In [ ]:
## Rest
S7_flux, S7_growth = community_pfba(S7, maize_root_medium, frac=1)
S7_flux_n, S7_growth_n = community_pfba(S7, medium_combined_naveed, frac=1)
S7_flux_a, S7_growth_a = community_pfba(S7, maize_root_medium_all, frac=1)
S7_growth_a
S7_growth
S7_growth_n
C7_flux, C7_growth = community_pfba(syncom, maize_root_medium)
C7_growth
C7_df_reset = C7_flux.reset_index().rename(columns={"reaction": "rxn_id"})
S7_df_reset = S7_flux.reset_index().rename(columns={"reaction": "rxn_id"})
S7_df_reset_n = S7_flux_n.reset_index().rename(columns={"reaction": "rxn_id"})
merged = S7_df_reset_n.merge(S7_df_reset, on="rxn_id", suffixes=("_S7n", "_S7"))

In [ ]:
# Compare medium fluxes
merged["medium_diff"] = merged["medium_S7n"] - merged["medium_S7"]

# Compare per-organism fluxes (excluding the dropped strain)
for strain in [col for col in C7_flux.columns if col != "AA7" and col != "medium"]:
    merged[f"{strain}_diff"] = merged[f"{strain}_S7n"] - merged[f"{strain}_S7"]

# Find where changes happened
diff_cols = [c for c in merged.columns if c.endswith("_diff")]
# Sum the absolute differences for each reaction
flux_changes = merged[diff_cols].abs().sum(axis=1)
flux_changes.name = "total_abs_diff"  # give it a proper name

# Filter for reactions with changes
changed_rxns = merged.loc[flux_changes > 1e-6, ["rxn_id"] + diff_cols]

# Add the diff magnitude as a column for sorting
changed_rxns["total_abs_diff"] = flux_changes[flux_changes > 1e-6]

# Sort by magnitude
changed_rxns = changed_rxns.sort_values(by="total_abs_diff", ascending=False)
# Identify the AA columns (excluding medium)
aa_cols = [c for c in C7_flux.columns if c.startswith("AA")]

# Filter: only EX_ reactions
mask_ex = C7_flux.index.str.startswith("EX_")

# Condition 1: AA7 > 0
mask_aa7_pos = C7_flux["AA7"] > 0

# Condition 2: all other AA's < 0
other_aas = [c for c in aa_cols if c != "AA7"]
mask_others_neg = (C7_flux[other_aas] < 0).all(axis=1)

# Combine masks
mask = mask_ex & mask_aa7_pos & mask_others_neg

# Get the matching reactions
aa7_only_prod = C7_flux.loc[mask]

print(aa7_only_prod.index)

In [ ]:
get_rxn_unknown(models, "HEX7")
get_met(AA2, "fru_c")
get_met(AA5, "f6p_c")
AA1.reactions.HEX7
AA1.metabolites.fru_c
AA5.metabolites.fru_c
get_met(AA5, "fru_c")
get_met(AA4, "f6p_c")
budget_plot_allInOne2(S7, "^f6p_c__AA4", maize_root_medium, print_rxns=True)
budget_plot_allInOne2(S7, "^f6p_c__AA4", medium_combined_naveed, print_rxns=True)
heatmap_fluxes_withinCommunity()
get_met(AA7, "f6p_c")
get_met(AA1, "f6p_c")
get_rxn(AA4, "ALLK")
get_rxn(AA4, "ALLULPE")
get_rxn_unknown(models, "ATPS4rpp")
get_rxn_unknown(models, "GPDDA1_1")
get_met_unknown(models, "g3pc_e")
get_rxn_unknown(models, "EX_g3pc_e")
get_met_unknown(models, "chol_e")

which reactions have same flux value?

In [ ]:
fluxies = convert_cooptradeoff_into_fluxes(S7, maize_root_medium)
fluxies2 = convert_cooptradeoff_into_fluxes(S7, medium_combined_naveed)
# list of columns to compare
cols = ["AA1_diff", "AA3_diff", "AA4_diff", "AA5_diff", "AA6_diff"]

# check if all values in these columns are the same row-wise
mask = changed_rxns[cols].eq(changed_rxns[cols].iloc[:, 0], axis=0).all(axis=1) & changed_rxns[cols].notna().any(axis=1)
# get the rows where they are all equal
same_rows = changed_rxns[mask]

print(f"Number of rows where all {cols} are the same: {mask.sum()}")

In [ ]:
#write_sbml_model(S7, "../S7.xml")

In [ ]:
# 1. Drop zero fluxes
nonzero_flux = fluxies2[fluxies2 != 0]

# round to e.g. 3 decimals before grouping
rounded_flux = nonzero_flux.round(5).abs()

# group by rounded flux
grouped = rounded_flux.groupby(rounded_flux).apply(lambda x: list(x.index))

# counts
counts = rounded_flux.value_counts()

# combine into a single DataFrame
result = pd.DataFrame({
    "count": counts,
    "reactions": grouped
}).reset_index().rename(columns={"index": "flux"})

# list of required reactions
required = ["AA1", "AA3", "AA4", "AA5", "AA6"]

In [ ]:
def contains_all(reaction_list):
    return all(any(req in r for r in reaction_list) for req in required)

In [ ]:
# filter rows where all required IDs are present
filtered2 = result[result["reactions"].apply(contains_all)]

get_rxn_unknown(models, "CTPS1")
get_met(AA1, "nh4_c")
budget_plot_allInOne(S7, "^nh4_c__AA3", maize_root_medium)
AA1.reactions.Growth.metabolites
test_medium(S7, maize_root_medium)
get_rxn(AA6, "PYK2")

## new Investigation, september

In [ ]:

def check_early_biomass_component2(model, medium_dict, check_rxn_id):
    rxn_id = "objective_check"
    sad_mets = []

    check_mets = model.reactions.get_by_id(check_rxn_id).metabolites
    #check_mets = model.reactions.check_rxn_id.metabolites
    for met, flux in check_mets.items():
        if flux < 0:  # only look at mets that are consumed
            #print(met)
            stoich = {met: flux}

            if rxn_id in model.reactions:  # biomass test reaction is updated
                rxn = model.reactions.get_by_id(rxn_id)
                rxn.subtract_metabolites(rxn.metabolites)
                rxn.add_metabolites(stoich)
            else:  # first time reaction is created
                new_rxn = Reaction(id=rxn_id, name="objective reaction", lower_bound=0, upper_bound=1000)
                new_rxn.add_metabolites(stoich)
                model.add_reactions([new_rxn])
                rxn = model.reactions.get_by_id(rxn_id)
                rxn.global_id = rxn_id
                rxn.community_id = "AA"

            model.objective = rxn_id

            with model:
                #change_medium(model, medium_dict)
                try:
                    pfba_flux = convert_cooptradeoff_into_fluxes(model, medium_dict)
                    if pfba_flux[rxn_id] == 0:
                        sad_mets.append(met.id)
                except Infeasible:
                    print("Cannot get result because pfba is infeasible")

    if rxn_id in model.reactions:
        rxn = model.reactions.get_by_id(rxn_id)
        model.remove_reactions([rxn])

    print(sad_mets)


In [ ]:
with S7_noS:
    _, growth_val = community_pfba(S7_noS, maize_root_medium)
    print(growth_val)
with S7_noS:
    S7_noS.reactions.SUCpts__AA1.knock_out()
    S7_noS.reactions.SUCpts__AA2.knock_out()
    #S7_2.reactions.SUCpts__AA6.knock_out()
    _, growth_val = community_pfba(S7_noS, maize_root_medium)
    print(growth_val)
with S7_noS:
    S7_noS.reactions.EX_sucr_e__AA1.knock_out()
    _, growth_val = community_pfba(S7_noS, maize_root_medium)
    print(growth_val)
with S7_noS:
    S7_noS.reactions.EX_sucr_e__AA4.knock_out()
    _, growth_val = community_pfba(S7_noS, maize_root_medium)
    print(growth_val)
with S7_2:
    S7_2.reactions.SUCpts__AA1.knock_out()
    _, growth_val = community_pfba(S7_2, maize_root_medium)
    print(growth_val)
get_met(S7_2, "sucr_e__AA1")
with S7:
    #S7_2.reactions.EX_sucr_e__AA2.knock_out()
    growth_val = test_medium(S7, medium_combined_naveed)
    print(growth_val)
get_met(S7, "sucr_e__AA2")
with S7_2:
    #S7_2.reactions.EX_sucr_e__AA1.knock_out()
    #S7_2.reactions.EX_sucr_e__AA3.knock_out()
    #S7_2.reactions.EX_sucr_e__AA5.knock_out()
    #S7_2.reactions.EX_sucr_e__AA6.knock_out()
    growth_val = test_medium(S7_2, medium_combined_naveed)
    print(growth_val)
with S7_2:
    growth_val = test_medium(S7_2, maize_root_medium)
    print(growth_val)
S7_2.metabolites.pi_c__AA1
S7_2.metabolites.fru_c__AA1
get_met(S7_2, "fru_c__AA1")
with S7_2:
    S7_2.reactions.EX_sucr_e__AA1.knock_out()
    S7_2.reactions.SUCpts__AA1.knock_out()
    S7_2.reactions.SPP__AA1.knock_out()
    S7_2.reactions.SUCR__AA1.knock_out()
    growth_val = test_medium(S7_2, maize_root_medium)
    print(growth_val)
with S7_2:
    S7_2.reactions.SPP__AA1.knock_out()
    growth_val = test_medium(S7_2, maize_root_medium)
    print(growth_val)
with S7_2:
    S7_2.reactions.SUCpts__AA1.knock_out()
    growth_val = test_medium(S7_2, maize_root_medium)
    print(growth_val)
with S7_2:
    S7_2.reactions.SUCR__AA1.knock_out()
    growth_val = test_medium(S7_2, maize_root_medium)
    print(growth_val)
with S7_2:
    #S7_2.reactions.EX_sucr_e__AA1.knock_out()
    #S7_2.reactions.EX_sucr_e__AA2.knock_out()
    S7_2.reactions.EX_sucr_e__AA3.knock_out()
    #S7_2.reactions.EX_sucr_e__AA4.knock_out()
    S7_2.reactions.EX_sucr_e__AA5.knock_out()
    S7_2.reactions.EX_sucr_e__AA6.knock_out()
    growth_val = test_medium(S7_2, maize_root_medium)
    print(growth_val)
with S7_2:
    S7_2.reactions.EX_sucr_e__AA1.knock_out()
    growth_val = test_medium(S7_2, maize_root_medium)
    print(growth_val)
with S7_2:
    #S7_2.reactions.EX_sucr_e__AA1.knock_out()
    growth_val = test_medium(S7_2, maize_root_medium)
    print(growth_val)
with S7_2:
    S7_2.reactions.SUCpts__AA1.knock_out()
    growth_val = test_medium(S7_2, maize_root_medium)
    print(growth_val)

In [ ]:
budget_plot_allInOne(S7_2, "^sucr_m", maize_root_medium)
get_met(AA3_noS, "sucr_e")
krumbach_com = e_to_m(maize_root_medium)
krumbach_com = {
    rxn: bound for rxn, bound in krumbach_com.items()
    if rxn in S7_2.reactions
}

In [ ]:
with S7_2:
    #S7_2.reactions.EX_sucr_e__AA1.knock_out()
    #S7_2.reactions.EX_sucr_e__AA2.knock_out()
    S7_2.reactions.EX_sucr_e__AA3.knock_out()
    #S7_2.reactions.EX_sucr_e__AA4.knock_out()
    S7_2.reactions.EX_sucr_e__AA5.knock_out()
    S7_2.reactions.EX_sucr_e__AA6.knock_out()
    S7_2.medium = krumbach_com
    basic_fluxes = S7_2.cooperative_tradeoff(fluxes=True, pfba=True).fluxes.transpose()
basic_fluxes
with S7_2:
    #S7_2.reactions.EX_sucr_e__AA1.knock_out()
    S7_2.reactions.SPP__AA1.knock_out()
    #S7_2.reactions.EX_sucr_e__AA2.knock_out()
    S7_2.reactions.EX_sucr_e__AA3.knock_out()
    #S7_2.reactions.EX_sucr_e__AA4.knock_out()
    S7_2.reactions.EX_sucr_e__AA5.knock_out()
    S7_2.reactions.EX_sucr_e__AA6.knock_out()
    print(test_medium(S7_2, krumbach_com))
get_rxn(S7_2, "SUCR__AA1")
get_met(S7_2, "suc6p_c__AA1")
get_met(S7_2, "sucr_e__AA1")
get_rxn(S7_2, "SUCpts__AA1")
with S7_2:
    #S7_2.reactions.EX_sucr_e__AA1.knock_out()
    #S7_2.reactions.EX_sucr_e__AA2.knock_out()
    S7_2.reactions.EX_sucr_e__AA3.knock_out()
    #S7_2.reactions.EX_sucr_e__AA4.knock_out()
    S7_2.reactions.EX_sucr_e__AA5.knock_out()
    S7_2.reactions.EX_sucr_e__AA6.knock_out()
    budget_plot_allInOne(S7_2, "^sucr_e__AA1", maize_root_medium, print_rxns=True)

In [ ]:
budget_plot_allInOne(S7_2, "^pyr_c__AA1", maize_root_medium, print_rxns=True)
budget_plot_allInOne(S7, "^fru_c__AA1", maize_root_medium, print_rxns=True)
budget_plot_allInOne(S7, "^fru_c__AA3", maize_root_medium, print_rxns=True)
budget_plot_allInOne(S7_2, "^fru_c__AA3", maize_root_medium)
budget_plot_allInOne(S7_2, "^sucr_c__AA3", maize_root_medium)
budget_plot_allInOne(S7_2, "^sucr_e__AA3", maize_root_medium)
budget_plot_allInOne(S7_2, "^sucr_c__AA4", maize_root_medium)
get_met(S7_2, "sucr_c__AA4")
get_met(S7_2, "sucr_p__AA4")
budget_plot_allInOne(S7_2, "^sucr_p__AA4", maize_root_medium)

# Spielplatz

In [ ]:
from cobra import Reaction


def check_early_biomass_component2(model, medium_dict, check_rxn_id):
    rxn_id = "objective_check"
    sad_mets = []

    check_mets = model.reactions.get_by_id(check_rxn_id).metabolites
    #check_mets = model.reactions.check_rxn_id.metabolites
    for met, flux in check_mets.items():
        if flux < 0:  # only look at mets that are consumed
            #print(met)
            stoich = {met: flux}

            if rxn_id in model.reactions:  # biomass test reaction is updated
                rxn = model.reactions.get_by_id(rxn_id)
                rxn.subtract_metabolites(rxn.metabolites)
                rxn.add_metabolites(stoich)
            else:  # first time reaction is created
                new_rxn = Reaction(id=rxn_id, name="objective reaction", lower_bound=0, upper_bound=1000)
                new_rxn.add_metabolites(stoich)
                model.add_reactions([new_rxn])

            model.objective = rxn_id

            with model:
                #change_medium(model, medium_dict)
                try:
                    pfba_flux = convert_cooptradeoff_into_fluxes(model, medium_dict)
                    if pfba_flux[rxn_id] == 0:
                        sad_mets.append(met.id)
                except Infeasible:
                    print("Cannot get result because pfba is infeasible")

    if rxn_id in model.reactions:
        rxn = model.reactions.get_by_id(rxn_id)
        model.remove_reactions([rxn])

    print(sad_mets)

In [ ]:
test = pfba(AA1)
test["Growth"]
test2 = convert_cooptradeoff_into_fluxes(S7, maize_root_medium)
test2_frame = test2.to_frame(name="fluxes")  # rename column to "fluxes"
test2_frame.index.name = None  # remove index name if there is one

test_frame = test.to_frame()
fluxes = get_pfba_fluxes(S7, maize_root_medium)

In [ ]:
def get_identity_model(model):
    """
    Gets you the info if we have community or individual model.
    :param model: cobrapy model
    :return: "ind" or "com"
    """
    ind_model_ids = [f"AA{i}" for i in range(1, 8)] + [f"AA{i}f" for i in range(1, 8)]
    if model.id in ind_model_ids:
        return "ind"
    else:
        return "com"


def budget_plot_allInOne2(model, query_term, medium, print_rxns=False):
    budget_mets, solution_frame = fba_and_query(model, query_term, medium)
    if budget_mets is None:
        return "Infeasible or no Growth; no data to plot"
    all_reactions = calc_producer_consumer(model, budget_mets, solution_frame)

    if print_rxns:
        print(all_reactions)

    budget_plot = make_budget_plot(all_reactions, budget_mets)